# 🪰 Drosophila Connectome Pong: TPU v6e-1 (Trillium) & A100 / L4 GPU Lab

### Biologisch-realistische Reinforcement-Learning Simulation zweier virtueller Fruchtfliegen-Gehirne (*Drosophila melanogaster*)
Basierend auf dem vollständigen Konnektom des Fliegenhirns (**FlyWire Consortium / Nature Oktober 2024**).

### ⚡ Hardware-Unterstützung (Automatische Erkennung):
- **Google TPU v6e-1 (Trillium)**: Google neueste TPU-Generation (bis zu 4,7x schnellerer Matrix-Durchsatz als v5e).
- **NVIDIA A100-SXM4 / PCIe (40GB/80GB)**: Höchste CUDA/Tensor-Core Leistung für parallele Simulationen.
- **NVIDIA L4 GPU**: Ada-Lovelace Architektur mit hoher Effizienz.

### 🔬 Features:
- **60 FPS WebSocket Cyberpunk Web-App (Kein Gradio)** mit animierten Fliegen (Flügelschlag nach Erregung).
- **3D-Konnektom-Gehirnkarten (Three.js WebGL)**: *Happy* (PAM Dopamin in Smaragdgrün) vs. *Sauer* (PPL1 Stress in Karminrot).
- **20-Minuten Auto-Checkpointing**: Sichert trainierte Fliegenhirne alle 20 Minuten automatisch auf Google Drive.

## 1. Hardware & Beschleuniger-Erkennung (TPU v6e-1 / A100 / L4)

In [ ]:
# Automatische Prüfung des aktiven Colab-Beschleunigers
import os, sys

# Check for GPU
!nvidia-smi 2>/dev/null || echo "[Info] Kein NVIDIA GPU Treiber aktiv (Prüfe TPU...)"

# Check for TPU
tpu_detected = 'COLAB_TPU_ADDR' in os.environ or 'TPU_NAME' in os.environ
if tpu_detected:
    print("\n✅ GOOGLE TPU BESCHLEUNIGER ERKANNT (v6e-1 Trillium oder v5e)!")
else:
    print("\n🚀 GPU oder CPU Modus aktiv.")

## 2. Google Drive einbinden (für 20-Minuten-Dauerspeicherung)

In [ ]:
# Mountet Google Drive, damit trainierte Fliegenhirne dauerhaft erhalten bleiben
from google.colab import drive
drive.mount('/content/drive')

import os
ckpt_dir = '/content/drive/MyDrive/fly_brain_checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)
print(f"[Drive] Checkpoint-Verzeichnis bereit: {ckpt_dir}")

## 3. Smarte Installation (TPU v6e-1 oder A100/L4 CUDA)

In [ ]:
# Installiert automatisch die optimalen Treiberpakete für TPU oder GPU
import os, subprocess

has_gpu = False
try:
    res = subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if res.returncode == 0:
        has_gpu = True
except:
    has_gpu = False

if has_gpu:
    print("⚡ Installiere JAX mit CUDA 12 Unterstützung für NVIDIA A100 / L4...")
    !pip install --upgrade -q "jax[cuda12]" flax optax safetensors pillow fastapi uvicorn[standard] websockets
else:
    print("⚡ Installiere JAX mit TPU Unterstützung für Google TPU v6e-1 (Trillium)...")
    !pip install --upgrade -q "jax[tpu]" flax optax safetensors pillow fastapi uvicorn[standard] websockets

import jax
print(f"\n[JAX Engine Aktiv]: {jax.devices()}")

## 4. Repository clonen & betreten

In [ ]:
!git clone https://github.com/finnytech/fly-brain-pong-tpu.git
%cd fly-brain-pong-tpu
!git pull

## 5. Öffentlichen Web-Host Tunnel & Simulation starten
Startet den WebSocket-Server auf Port 8000 und öffnet den sicheren öffentlichen Tunnel für den Browser.

In [ ]:
# 1. Cloudflare Quick Tunnel starten (kostenlos, sicher, sofortiger HTTPS Link)
!curl -s https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb && dpkg -i cloudflared.deb > /dev/null
!cloudflared tunnel --url http://localhost:8000 > tunnel.log 2>&1 &
import time; time.sleep(4)
!grep -o 'https://.*\.trycloudflare\.com' tunnel.log | head -n 1

# 2. Hauptskript auf TPU v6e-1 oder A100 starten
!python main.py --port 8000 --checkpoint-interval 20.0

## 6. Gespeicherte 20-Minuten Checkpoints einsehen

In [ ]:
import glob, json
checkpoints = glob.glob('/content/drive/MyDrive/fly_brain_checkpoints/*.json')
print(f"Gefundene Checkpoint-Metadaten ({len(checkpoints)}):")
for cp in sorted(checkpoints):
    with open(cp) as f:
        meta = json.load(f)
        print(f"- {cp}: Step {meta.get('step')}, Score: {meta.get('score1')}:{meta.get('score2')}, Fliege 1 Wins: {meta.get('fly1_total_wins')}, Fliege 2 Wins: {meta.get('fly2_total_wins')}")